### Tuning top_k, chunk_size, chunk_overlap

Simple grid search over `top_k` in `[4, 6, 8]`, `chunk_size` in `[100, 500, 700]`,
and `chunk_overlap` in `[50, 100, 150]` using `RecursiveCharacterTextSplitter`,
scored against generated labeled query set in `eval_queries.json`
(`query` + `relevant_pages`).

No LLM used instead tested on retriveal quality.

Combinations where `chunk_overlap >= chunk_size` are skipped.

In [ ]:
import os
import sys
import json
from itertools import product

import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer, util
from langchain_text_splitters import RecursiveCharacterTextSplitter
import fitz  # PyMuPDF

sys.path.append(os.path.abspath(".."))
load_dotenv("../.env")

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
PDF_PATH = "../instructions/AWS_Customer_Agreement.pdf"

model = SentenceTransformer(EMBEDDING_MODEL)
EMBEDDING_MODEL

e:\india_project\rag_aws_\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Generating `eval_queries.json`
generating the eval set to test the quality of the retriveal and find the optimal size for chunks, overlap and top k values. 
structure for the eval set : 
`query:<query>, relevant_pages: <pages>, difficulty:<level>`

In [ ]:
from openai import OpenAI

def generate_eval_queries(pdf_path, n_queries=20, output_path="../eval_queries_generated.json"):
    doc = fitz.open(pdf_path)
    pages_text = [
        f"[PAGE {i}]\n{page.get_text('text').strip()}"
        for i, page in enumerate(doc, start=1)
        if page.get_text("text").strip()
    ]
    full_text = "\n\n".join(pages_text)

    prompt = f"""You are creating an evaluation dataset for a RAG system.

Generate exactly {n_queries} factual questions from the document below that:
1. Are answerable from a SPECIFIC page (record which page number(s) in "relevant_pages")
2. Require looking up a specific fact, clause, term, or number
3. Cannot be answered from general knowledge alone
4. Include 2-3 trap questions that sound plausible but are NOT answered anywhere in
   the document (use "relevant_pages": [] for those)

Document:
{full_text}

Respond with ONLY a valid JSON array, no preamble or markdown fences:
[{{"query": "...", "relevant_pages": [1], "difficulty": "easy|medium|hard|trap"}}]"""

    client = OpenAI(base_url=os.getenv("BASE_URL"), api_key=os.getenv("OPENAI_API_KEY"))
    response = client.chat.completions.create(
        model=os.getenv("MODEL_NAME"),
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )

    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    queries = json.loads(raw)

    with open(output_path, "w") as f:
        json.dump(queries, f, indent=2)
    return queries


generated = generate_eval_queries(PDF_PATH)
len(generated)

3

In [6]:
with open("../eval_queries.json") as f:
    EVAL_QUERIES = json.load(f)

len(EVAL_QUERIES)

24

In [7]:
TOP_K_VALUES = [4, 6, 8]
CHUNK_SIZES = [100, 500, 700]
CHUNK_OVERLAPS = [50, 100, 150]


def load_pdf_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()
        if text:
            pages.append({"text": text, "page": page_num})
    return pages


def chunk_pages(pages, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " "],
    )
    chunks = []
    for page_data in pages:
        for split in splitter.split_text(page_data["text"]):
            chunks.append({"text": split, "page": page_data["page"]})
    return chunks


pages = load_pdf_pages(PDF_PATH)
len(pages)

19

In [ ]:
def retrieve_topk(chunk_embeddings, chunks, query, top_k):
    q_emb = model.encode([query])
    scores = util.cos_sim(q_emb, chunk_embeddings)[0].numpy()
    top_idx = scores.argsort()[::-1][:top_k]
    return [chunks[i]["page"] for i in top_idx]


def score_config(chunks, chunk_embeddings, queries, top_k):
    """Average Recall@k and MRR across all queries for one (chunk_size, overlap, top_k)."""
    recalls, mrrs = [], []
    for q in queries:
        relevant = set(q["relevant_pages"])
        retrieved_pages = retrieve_topk(chunk_embeddings, chunks, q["query"], top_k)

        recalls.append(len(relevant & set(retrieved_pages)) / len(relevant) if relevant else 0.0)

        rr = 0.0
        for rank, page in enumerate(retrieved_pages, start=1):
            if page in relevant:
                rr = 1.0 / rank
                break
        mrrs.append(rr)

    recall = sum(recalls) / len(recalls)
    mrr = sum(mrrs) / len(mrrs)
    return recall, mrr

In [6]:
results = []

for chunk_size, chunk_overlap in product(CHUNK_SIZES, CHUNK_OVERLAPS):
    if chunk_overlap >= chunk_size:
        continue  # invalid combination, skip

    chunks = chunk_pages(pages, chunk_size, chunk_overlap)
    chunk_embeddings = model.encode([c["text"] for c in chunks], batch_size=32, show_progress_bar=False)

    for top_k in TOP_K_VALUES:
        recall, mrr = score_config(chunks, chunk_embeddings, EVAL_QUERIES, top_k)
        results.append({
            "chunk_size": chunk_size,
            "chunk_overlap": chunk_overlap,
            "top_k": top_k,
            "num_chunks": len(chunks),
            "recall_at_k": round(recall, 4),
            "mrr": round(mrr, 4),
            "composite": round(0.5 * recall + 0.5 * mrr, 4),
        })

results_df = pd.DataFrame(results).sort_values("composite", ascending=False).reset_index(drop=True)
results_df

C:\Users\yashd\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1790: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


,chunk_size,chunk_overlap,top_k,num_chunks,recall_at_k,mrr,composite
0,700,100,8,109,0.8958,0.8080,0.8519
1,500,100,8,159,0.8958,0.7965,0.8462
2,500,100,6,159,0.8958,0.7965,0.8462
3,700,150,8,117,0.8750,0.8021,0.8385
4,700,150,4,117,0.8542,0.8021,0.8281
5,700,150,6,117,0.8542,0.8021,0.8281
6,500,100,4,159,0.8542,0.7882,0.8212
7,700,100,6,109,0.8333,0.8021,0.8177
8,700,100,4,109,0.8333,0.8021,0.8177
9,500,50,8,143,0.8542,0.7285,0.7913


In [7]:
print("Top 5 configurations:")
results_df.head(5)

Top 5 configurations:


,chunk_size,chunk_overlap,top_k,num_chunks,recall_at_k,mrr,composite
0,700,100,8,109,0.8958,0.8080,0.8519
1,500,100,8,159,0.8958,0.7965,0.8462
2,500,100,6,159,0.8958,0.7965,0.8462
3,700,150,8,117,0.8750,0.8021,0.8385
4,700,150,4,117,0.8542,0.8021,0.8281
